# District Specialty Demand Exploration

Explore **NFHS-5 district health indicators** and map them to the 15 hackathon specialty categories using `dais_2026.hackathon.specialty_category_mapping`.

**Goal:** For each district, identify which specialty categories show the highest demand signals.

| Table | Role |
|---|---|
| `databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators` | District-level demand indicators (NFHS-5) |
| `dais_2026.hackathon.specialty_category_mapping` | Maps NFHS indicator columns → specialty category |

In [3]:
from databricks.connect import DatabricksSession
from databricks.sdk.core import Config

config = Config(
    profile="stoic_pre"
)

spark = DatabricksSession.builder.sdkConfig(config).getOrCreate()


In [4]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType

NFHS_TABLE = (
    "databricks_virtue_foundation_dataset_dais_2026"
    ".virtue_foundation_dataset.nfhs_5_district_health_indicators"
)
MAPPING_TABLE = "dais_2026.hackathon.specialty_category_mapping"

CATEGORIES = [
    "Primary Care & General Medicine",
    "Cardiovascular Care",
    "Mental Health & Behavioral Health",
    "Neurology & Neurosciences",
    "Cancer Care (Oncology)",
    "Women\u2019s Health",
    "Pediatrics & Child Health",
    "Orthopedics & Musculoskeletal Care",
    "Respiratory & Pulmonary Care",
    "Endocrine & Metabolic Care",
    "Eye Care (Ophthalmology)",
    "Dental & Oral Health",
    "Dermatology & Skin Care",
    "Emergency, Critical Care & Anesthesia",
    "Surgery & Specialty Procedures",
]

GEO_COLUMNS = {
    "district_name",
    "state_ut",
    "households_surveyed",
    "women_15_49_interviewed",
}

def normalize_category(col):
    """Normalize punctuation so mapping-table labels match the canonical list."""
    return F.regexp_replace(F.trim(col), "\u2019", "'")

print(f"NFHS table: {NFHS_TABLE}")
print(f"Mapping table: {MAPPING_TABLE}")
print(f"Expected categories: {len(CATEGORIES)}")

NFHS table: databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators
Mapping table: dais_2026.hackathon.specialty_category_mapping
Expected categories: 15


## 1. Inspect source tables

In [5]:
mapping_raw = spark.table(MAPPING_TABLE)
nfhs_raw = spark.table(NFHS_TABLE)

print("=== specialty_category_mapping schema ===")
mapping_raw.printSchema()
display(mapping_raw.limit(20))

print("=== nfhs_5_district_health_indicators schema ===")
nfhs_raw.printSchema()
display(nfhs_raw.select("district_name", "state_ut", "households_surveyed").limit(10))

print(f"Districts in NFHS: {nfhs_raw.select('district_name', 'state_ut').distinct().count()}")
print(f"Mapping rows: {mapping_raw.count()}")

=== specialty_category_mapping schema ===
root
 |-- specialties: string (nullable = true)
 |-- category: string (nullable = true)



DataFrame[specialties: string, category: string]

=== nfhs_5_district_health_indicators schema ===
root
 |-- district_name: string (nullable = true)
 |-- state_ut: string (nullable = true)
 |-- households_surveyed: double (nullable = true)
 |-- women_15_49_interviewed: double (nullable = true)
 |-- men_15_54_interviewed: double (nullable = true)
 |-- female_population_age_6_years_and_above_ever_schooled_pct: double (nullable = true)
 |-- population_below_age_15_years_pct: double (nullable = true)
 |-- sex_ratio_total_f_per_1000_m: double (nullable = true)
 |-- sex_ratio_at_birth_5y_f_per_1000_m: string (nullable = true)
 |-- child_u5_whose_birth_was_civil_reg_pct: double (nullable = true)
 |-- deaths_in_the_last_3_years_civil_reg_pct: string (nullable = true)
 |-- hh_electricity_pct: double (nullable = true)
 |-- hh_improved_water_pct: double (nullable = true)
 |-- hh_use_improved_sanitation_pct: double (nullable = true)
 |-- households_using_clean_fuel_for_cooking_pct: double (nullable = true)
 |-- households_using_iodized_salt_pct: 

DataFrame[district_name: string, state_ut: string, households_surveyed: double]

Districts in NFHS: 706
Mapping rows: 2935


In [7]:
def resolve_mapping_columns(df):
    """Infer indicator and category column names from the mapping table."""
    cols = df.columns
    lower = {c: c.lower() for c in cols}

    category_candidates = [
        c for c in cols
        if any(k in lower[c] for k in ["category", "specialty_category", "specialty"])
        and "indicator" not in lower[c]
    ]
    indicator_candidates = [
        c for c in cols
        if any(k in lower[c] for k in ["indicator", "column", "nfhs", "field", "metric"])
    ]

    if not category_candidates:
        raise ValueError(f"Could not find category column in mapping table: {cols}")
    if not indicator_candidates:
        raise ValueError(f"Could not find indicator column in mapping table: {cols}")

    category_col = category_candidates[0]
    indicator_col = indicator_candidates[0]
    return indicator_col, category_col


indicator_col, category_col = resolve_mapping_columns(mapping_raw)
print(f"Using indicator column: {indicator_col}")
print(f"Using category column:  {category_col}")

mapping = (
    mapping_raw
    .select(
        F.trim(F.col(indicator_col)).alias("indicator_column"),
        normalize_category(F.col(category_col)).alias("category"),
    )
    .filter(F.col("indicator_column").isNotNull() & F.col("category").isNotNull())
    .distinct()
)

display(mapping.groupBy("category").count().orderBy(F.desc("count")))

canonical_categories = {c.replace("\u2019", "'") for c in CATEGORIES}
mapped_categories = {
    row.category.replace("\u2019", "'")
    for row in mapping.select("category").distinct().collect()
}
missing_from_mapping = sorted(canonical_categories - mapped_categories)
unexpected_in_mapping = sorted(mapped_categories - canonical_categories)

print("Categories missing from mapping table:", missing_from_mapping or "none")
print("Unexpected categories in mapping table:", unexpected_in_mapping or "none")

ValueError: Could not find indicator column in mapping table: ['specialties', 'category']

## 2. Clean NFHS indicator values

Many NFHS percentage columns are stored as strings with special values (`*`, parentheses, blanks). We parse them to numeric values before aggregation.

In [ ]:
@F.udf("struct<value:double, is_suppressed:boolean, raw_value:string>")
def parse_nfhs_pct(raw):
    if raw is None:
        return (None, False, None)
    text = str(raw).strip()
    if text == "":
        return (None, False, text)
    if text == "*":
        return (None, True, text)
    cleaned = text.replace("(", "").replace(")", "")
    try:
        return (float(cleaned), False, text)
    except ValueError:
        return (None, False, text)


mapped_indicator_names = [
    row.indicator_column
    for row in mapping.select("indicator_column").distinct().collect()
]

nfhs_columns = set(nfhs_raw.columns)
available_indicators = [c for c in mapped_indicator_names if c in nfhs_columns]
missing_indicators = sorted(set(mapped_indicator_names) - nfhs_columns)

print(f"Mapped indicators available in NFHS table: {len(available_indicators)}")
print(f"Mapped indicators missing from NFHS table: {len(missing_indicators)}")
if missing_indicators:
    print("First 10 missing:", missing_indicators[:10])

In [ ]:
def build_unpivot_expr(indicator_names):
    if not indicator_names:
        raise ValueError("No mapped indicators found in the NFHS table.")
    parts = []
    for name in indicator_names:
        parts.append(f"'{name}'")
        parts.append(f"`{name}`")
    return f"stack({len(indicator_names)}, {', '.join(parts)})"


stack_expr = build_unpivot_expr(available_indicators)

nfhs_long = (
    nfhs_raw
    .select(
        F.trim(F.col("district_name")).alias("district_name"),
        F.trim(F.col("state_ut")).alias("state_ut"),
        F.col("households_surveyed").cast(DoubleType()).alias("households_surveyed"),
        F.expr(stack_expr).alias("indicator_column", "indicator_raw"),
    )
    .withColumn("parsed", parse_nfhs_pct(F.col("indicator_raw")))
    .withColumn("demand_pct", F.col("parsed.value"))
    .withColumn("is_suppressed", F.col("parsed.is_suppressed"))
    .drop("parsed")
)

display(
    nfhs_long
    .groupBy("is_suppressed")
    .agg(
        F.count("*").alias("rows"),
        F.avg("demand_pct").alias("avg_demand_pct"),
    )
)

display(nfhs_long.filter(F.col("demand_pct").isNotNull()).limit(15))

## 3. Join indicators to specialty categories

In [ ]:
canonical_categories = [c.replace("\u2019", "'") for c in CATEGORIES]

category_demand = (
    nfhs_long
    .join(mapping, on="indicator_column", how="inner")
    .filter(F.col("category").isin(canonical_categories))
)

district_category_scores = (
    category_demand
    .groupBy("district_name", "state_ut", "category")
    .agg(
        F.avg("demand_pct").alias("avg_demand_pct"),
        F.max("demand_pct").alias("max_demand_pct"),
        F.expr("percentile(demand_pct, 0.5)").alias("median_demand_pct"),
        F.count(F.when(F.col("demand_pct").isNotNull(), 1)).alias("indicator_count"),
        F.count(F.when(F.col("is_suppressed"), 1)).alias("suppressed_indicator_count"),
        F.first("households_surveyed", ignorenulls=True).alias("households_surveyed"),
    )
)

district_window = Window.partitionBy("district_name", "state_ut")

district_category_ranked = (
    district_category_scores
    .withColumn(
        "category_rank_in_district",
        F.dense_rank().over(
            district_window.orderBy(F.desc("avg_demand_pct"), F.desc("indicator_count"))
        ),
    )
    .withColumn(
        "demand_share_in_district",
        F.col("avg_demand_pct")
        / F.sum("avg_demand_pct").over(district_window),
    )
    .orderBy("state_ut", "district_name", "category_rank_in_district")
)

display(district_category_ranked.limit(30))

## 4. Top demand category per district

In [ ]:
top_category_per_district = (
    district_category_ranked
    .filter(F.col("category_rank_in_district") == 1)
    .select(
        "district_name",
        "state_ut",
        F.col("category").alias("top_demand_category"),
        "avg_demand_pct",
        "indicator_count",
        "households_surveyed",
    )
    .orderBy(F.desc("avg_demand_pct"))
)

display(top_category_per_district.limit(25))

print("=== How often each category is #1 demand in a district ===")
display(
    top_category_per_district
    .groupBy("top_demand_category")
    .agg(
        F.count("*").alias("districts_where_top"),
        F.round(F.avg("avg_demand_pct"), 2).alias("avg_top_demand_pct"),
    )
    .orderBy(F.desc("districts_where_top"))
)

## 5. State-level category demand summary

In [ ]:
state_category_summary = (
    district_category_ranked
    .groupBy("state_ut", "category")
    .agg(
        F.countDistinct("district_name").alias("district_count"),
        F.round(F.avg("avg_demand_pct"), 2).alias("avg_demand_pct"),
        F.round(F.max("avg_demand_pct"), 2).alias("max_district_avg_demand_pct"),
    )
    .orderBy("state_ut", F.desc("avg_demand_pct"))
)

display(state_category_summary)

## 6. Heatmap — top 3 categories per district (sample states)

Pivot the top 3 ranked categories for a quick visual scan. Adjust `SAMPLE_STATES` to focus on states of interest.

In [ ]:
import pandas as pd

SAMPLE_STATES = ["Maharashtra", "Uttar Pradesh", "Bihar", "Kerala"]

top3_pdf = (
    district_category_ranked
    .filter(F.col("state_ut").isin(SAMPLE_STATES))
    .filter(F.col("category_rank_in_district") <= 3)
    .select(
        "state_ut",
        "district_name",
        "category_rank_in_district",
        "category",
        F.round("avg_demand_pct", 1).alias("avg_demand_pct"),
    )
    .orderBy("state_ut", "district_name", "category_rank_in_district")
    .toPandas()
)

if top3_pdf.empty:
    print("No rows for sample states. Check state_ut spelling in NFHS data.")
else:
    top3_pdf["label"] = (
        top3_pdf["category"].str.slice(0, 28)
        + " ("
        + top3_pdf["avg_demand_pct"].astype(str)
        + "%)"
    )
    pivot = (
        top3_pdf
        .pivot_table(
            index=["state_ut", "district_name"],
            columns="category_rank_in_district",
            values="label",
            aggfunc="first",
        )
        .rename(columns={1: "#1 category", 2: "#2 category", 3: "#3 category"})
    )
    display(pivot.reset_index())

## 7. Drill into one district

Change `FOCUS_DISTRICT` and `FOCUS_STATE` to inspect a specific district.

In [ ]:
FOCUS_DISTRICT = "Mumbai"
FOCUS_STATE = "Maharashtra"

district_detail = (
    category_demand
    .filter(
        (F.col("district_name") == FOCUS_DISTRICT)
        & (F.col("state_ut") == FOCUS_STATE)
    )
    .select(
        "category",
        "indicator_column",
        "indicator_raw",
        F.round("demand_pct", 2).alias("demand_pct"),
        "is_suppressed",
    )
    .orderBy(F.desc("demand_pct"))
)

display(district_detail)

display(
    district_category_ranked
    .filter(
        (F.col("district_name") == FOCUS_DISTRICT)
        & (F.col("state_ut") == FOCUS_STATE)
    )
    .orderBy("category_rank_in_district")
)

## 8. Optional — save curated output

Uncomment to persist the district-category demand scores for downstream app or SQL use.

In [ ]:
# OUTPUT_TABLE = "dais_2026.hackathon.district_category_demand_scores"
# (
#     district_category_ranked
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(OUTPUT_TABLE)
# )
# print(f"Saved to {OUTPUT_TABLE}")